# SEE in Google Colab for output

* Transfer Learning: Using knowledge learned from one task
and applying it to a new related task. 
* instead of training from scratch we use the retrained model
#### VGG16: VGG16 is a pre-trained deep Convolutional Neural Network (CNN) used for image feature extraction.

* Developed by Oxford Visual Geometry Group (VGG)

* Trained on ImageNet dataset (1.2 million images, 1000 classes)

* Has 16 learnable layers (13 convolution + 3 fully connected)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator # it means loads images and applies data augumentation and feed to the dl model in batches
from glob import glob
import matplotlib.pyplot as plt

* preprocess_input : prepares images exactly like vgg16 expects

In [ ]:
# Image size
IMAGE_SIZE = [224, 224]
# dataset path
data_path = 'data'
# Load VGG16
vgg = VGG16(
    input_shape=IMAGE_SIZE + [3],
    weights='imagenet', # loads pretrained knowledge
    include_top=False # removes imagenet's classifier (1000 classes)
)

* VGG16 requires images of size 224 x 224
* input_shape=IMAGE_SIZE + [3] : It means it is an rgb image "3" indicates that the number of channels

In [ ]:
for layer in vgg.layers:
    layer.trainable=False

* above one tells that we don't train  the weights of the VGG16 layer because it already prre-trained this is heart of the model this step is called "transfer Learning"

In [ ]:
folders = glob(data_path + '/*') # used to read the folders as with_mask and without_mask

In [ ]:
x=Flatten()(vgg.output) 
# generally the VGG16 output as 3d feature maps it flatten then convert into 1d vector

In [ ]:
prediction=Dense(len(folders),activation='softmax')(x)
# it is dense layer with 2 neurons and using softmax for outputs probabilities as
# with_mask -> probability
# without_mask -> probability

In [ ]:
model=Model(inputs=vgg.input,outputs=prediction) # the input is the pretrained model data but the output is the predictions for the what we have data related to with_mask and without_mask
model.summary()

* the above model summary at last the dense_3 shows (None,2) tells that you cut the last layer and you have the 2 categories as with_mask and without_mask

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

### the categorical_crossentropy does this like:
| Class        | Label  |
| ------------ | ------ |
| with_mask    | [1, 0] |
| without_mask | [0, 1] |


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input, # normalize images for VGG16
    shear_range=0.2, # tilts image
    zoom_range=0.2, # zooms image
    horizontal_flip=True, # flips image
    validation_split=0.2 # 80% train, 20% validation
)


In [ ]:
training_set=train_datagen.flow_from_directory(
    data_path,
    target_size=(224,224), # resize every image to this specifications
    batch_size=32, # model sees 32 images at a time
    class_mode='categorical', 
    subset='training' # it means 80% for training 
)

* flow_from_directory : It reads images from folders, converts them into batches, assigns labels automatically, and feeds them to the neural network during training.

#### class_mode='categorical' : it means
| Folder       | Label    |
| ------------ | -------- |
| with_mask    | [1, 0] |
| without_mask | [0, 1] |



In [ ]:
validation_set = train_datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation' # 20% of data used for validation purpose
)

In [ ]:
r=model.fit(
    training_set,
    validation_data=validation_set,
    epochs=5
)

In [ ]:
plt.plot(r.history['loss'], label='train loss')
plt.plot(r.history['val_loss'], label='val loss')
plt.legend()
plt.show()

plt.plot(r.history['accuracy'], label='train accuracy')
plt.plot(r.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.show()
